**DeapSECURE module 2: Dealing with Big Data**

# Warm-up: Your First Pandas `DataFrame`, the Voyages of Columbus

<img src="fig/columbus-voyages.svg" alt="Map of the routes of Columbus's four voyages across the Atlantic, 1492-1504" width="760">

*The four voyages of Christopher Columbus, 1492–1504.
Map by Phirosiberia, [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Viajes_de_colon_en.svg),
[CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/).*

Before we work with large security datasets, let's warm up with a small table that is easy to check by eye:
the four voyages of Christopher Columbus to the Americas (1492–1504).

The data comes from the Wikipedia article
["Voyages of Christopher Columbus"](https://en.wikipedia.org/wiki/Voyages_of_Christopher_Columbus).
Where the sources disagree on a date, we use the date in the article's infobox.

**Slides:** [Pandas Warm-up: Columbus's Voyages](slides/Pandas-Warmup-Columbus.pdf) (PDF, 16 slides),
also as an editable [PowerPoint](slides/Pandas-Warmup-Columbus.pptx) with speaker notes.

**What you will practice:**
1. Creating a `DataFrame` from a Python dictionary
2. Inspecting it with `head()`, `shape`, `columns` and `dtypes`
3. Selecting columns and rows
4. Filtering rows with a condition
5. Sorting, and adding a computed column
6. Finding missing values (`NaN`)
7. Grouping and counting
8. Drawing a map from a DataFrame
9. Answering the same questions in Excel, and reading a CSV file

As in the other notebooks, `#TODO` marks the places where you fill in the code.
Suggested solutions are at the end of the notebook. Try each exercise yourself first!

In [ ]:
"""Run this cell to import pandas.""";
import pandas

## 1. Creating a `DataFrame`

A `DataFrame` is a 2-D table with **row labels** (the *index*) and **column labels**.
One easy way to create a `DataFrame` is from a dictionary:
each **key** becomes a column name, and each **list** holds that column's values.
Each list must have the same length, one value per row.

Our first table has one row per voyage:

In [ ]:
"""Create a DataFrame named VOYAGES with one row per voyage:""";

VOYAGES = pandas.DataFrame({
    "voyage":         [1, 2, 3, 4],
    "depart_date":    ["1492-08-03", "1493-09-24", "1498-05-30", "1502-05-12"],
    "return_date":    ["1493-03-04", "1496-06-11", "1500-08-23", "1504-11-07"],
    "depart_port":    ["Palos", "Cadiz", "Sanlucar", "Cadiz"],
    "num_ships":      [3, 17, 6, 4],
    "first_landfall": ["San Salvador", "Dominica", "Trinidad", "Martinique"],
})

VOYAGES

**QUESTIONS:**
* Which part of the output is the *index*? Which part are the *column labels*?
* We did not give pandas an index. What did it use by default?

## 2. Inspecting a `DataFrame`

These are the first things to check on any new dataset:

| Code | What it tells you |
|---|---|
| `df.head(n)` | the first `n` rows (default 5) |
| `df.shape` | `(number of rows, number of columns)` |
| `df.columns` | the column labels |
| `df.dtypes` | the data type of each column |
| `df.info()` | all of the above, plus missing-value counts |

In [ ]:
"""EXERCISE 2.1: Print the shape of VOYAGES""";

#TODO

In [ ]:
"""EXERCISE 2.2: Print the data type of each column""";

#TODO

**QUESTION:** What data type did pandas give `depart_date` and `return_date`?
Can we subtract one from the other to get the length of a voyage? (We will fix this in Section 5.)

## 3. Selecting Columns and Rows

* **One column:** `df["col"]` returns a `Series`.
* **Several columns:** `df[["col1", "col2"]]` returns a `DataFrame`. Note the *double* brackets.
* **Rows by label:** `df.loc[label]`
* **Rows by position:** `df.iloc[position]`

In [ ]:
"""EXAMPLE: select one column. What kind of object comes back?""";

print(VOYAGES["depart_port"])
print(type(VOYAGES["depart_port"]))

In [ ]:
"""EXERCISE 3.1: Select only the columns `voyage` and `num_ships`""";

#TODO

In [ ]:
"""EXERCISE 3.2: Show the row for the 3rd voyage (index label 2) with .loc""";

#TODO

## 4. A Bigger Table: Ships and Captains

Now let's build a second table with one row **per ship**.
We know the ships and captains of the first voyage well.
For the fourth voyage, the Wikipedia article names the ships but not all of their captains,
so we enter `None` there. Pandas will show these as missing values.

In [ ]:
"""Create the SHIPS DataFrame (one row per ship):""";

SHIPS = pandas.DataFrame({
    "voyage":    [1, 1, 1, 4, 4, 4, 4],
    "ship":      ["Santa Maria", "Pinta", "Nina",
                  "Capitana", "Gallega", "Vizcaina", "Santiago de Palos"],
    "ship_type": ["carrack", "caravel", "caravel",
                  "caravel", "caravel", "caravel", "caravel"],
    "captain":   ["Christopher Columbus", "Martin Alonso Pinzon", "Vicente Yanez Pinzon",
                  "Christopher Columbus", None, None, None],
    "owner_or_master": ["Juan de la Cosa", "Francisco Martin Pinzon", "Juan Nino",
                        None, None, None, None],
})

SHIPS

> **Historical note:** On the first voyage, Columbus commanded the fleet from the *Santa María*,
> which was owned by Juan de la Cosa. The *Niña*'s owner and master was Juan Niño.
> On the fourth voyage, Columbus sailed on the flagship *Capitana*.

### 4.1 Filtering Rows with a Condition

A comparison on a column gives a **boolean `Series`** of `True`/`False` values.
Put it inside `[]` to keep only the rows where it is `True`:

```python
SHIPS[SHIPS["voyage"] == 1]
```

In [ ]:
"""EXAMPLE: which ships sailed on the first voyage?""";

SHIPS[SHIPS["voyage"] == 1]

In [ ]:
"""EXERCISE 4.1: Show only the caravels""";

#TODO

In [ ]:
"""EXERCISE 4.2: Which ships did Columbus himself captain?
Hint: compare the `captain` column to "Christopher Columbus".""";

#TODO

### 4.2 Missing Values

`df.isna()` marks every missing cell with `True`.
Since `True` counts as 1, `df.isna().sum()` counts the missing values in each column.

In [ ]:
"""EXERCISE 4.3: Count the missing values in each column of SHIPS""";

#TODO

In [ ]:
"""EXERCISE 4.4: Show the ships whose captain is NOT known.
Hint: use .isna() on a single column as the filter.""";

#TODO

## 5. Dates, Sorting and New Columns

`pandas.to_datetime()` converts text such as `"1492-08-03"` to real dates.
Then we can subtract them: the result is a duration, and `.dt.days` turns it into a number of days.

In [ ]:
"""Run this cell to convert the date columns to real dates""";

VOYAGES["depart_date"] = pandas.to_datetime(VOYAGES["depart_date"])
VOYAGES["return_date"] = pandas.to_datetime(VOYAGES["return_date"])
VOYAGES.dtypes

In [ ]:
"""EXERCISE 5.1: Add a new column `duration_days` holding the length of each voyage in days.
Hint: (return_date - depart_date).dt.days""";

VOYAGES["duration_days"] = #TODO
VOYAGES

In [ ]:
"""EXERCISE 5.2: Sort the voyages from longest to shortest.
Hint: sort_values(..., ascending=False)""";

#TODO

**QUESTIONS:**
* Which voyage was the longest? Which was the shortest?
* The 4th voyage comes second, partly because the crew was stranded on Jamaica for about a year (June 1503 to June 1504).

### Plotting the Durations

*Finish Exercise 5.1 first: this plot uses the `duration_days` column.*

A table of four numbers is easy to read, but a chart makes the comparison clear at a glance.
Pandas works well with `matplotlib`: we pass DataFrame columns directly to the plotting functions.

A **horizontal bar chart** (`barh`) is a good choice for comparing a few quantities with text labels.
Bar *length* shows the duration. Sorting the bars makes the ranking easy to see.

In [ ]:
"""Run this cell to plot how long each voyage lasted""";
from matplotlib import pyplot
%matplotlib inline

# Sort shortest -> longest; barh draws from the bottom up, so the longest ends up on top
plot_data = VOYAGES.sort_values("duration_days")
labels = ("Voyage " + plot_data["voyage"].astype(str) + " ("
          + plot_data["depart_date"].dt.year.astype(str) + "-"
          + plot_data["return_date"].dt.year.astype(str) + ")")

fig, ax = pyplot.subplots(figsize=(8, 3.5))
bars = ax.barh(labels, plot_data["duration_days"], height=0.55, color="#2a78d6")

# Write the number of days at the end of each bar
ax.bar_label(bars, labels=[f"{d} days" for d in plot_data["duration_days"]],
             padding=6, color="#52514e", fontsize=10)

ax.set_title("Duration of Columbus's four voyages", loc="left", fontsize=13)
ax.set_xlabel("Days from departure to return")
ax.set_xlim(0, plot_data["duration_days"].max() * 1.18)   # leave room for the labels

# Keep the frame light, so the bars stand out
for side in ["top", "right", "left"]:
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color("#c3c2b7")
ax.tick_params(axis="y", length=0)
ax.tick_params(axis="x", colors="#52514e")
ax.xaxis.grid(True, color="#e6e5e0", linewidth=0.8)
ax.set_axisbelow(True)

pyplot.tight_layout()
pyplot.show()

**QUESTIONS:**
* Why did we sort the data before plotting? Try plotting `VOYAGES` without sorting it. Which version is easier to read?
* The first voyage was by far the shortest, yet it is the most famous.
  Does a longer voyage mean more places were visited? (You can check this in Section 6.)

In [ ]:
"""EXERCISE 5.3: Make a similar bar chart of `num_ships` for each voyage.
Hint: copy the cell above and change the column you plot.""";

#TODO

## 6. Grouping: Places Visited

Our last table has one row per **landfall** (a place where the fleet arrived), with dates from the article.
A tidy, one-row-per-event table like this is the shape most real datasets have.
Security logs, for example, have one row per connection or login.

In [ ]:
"""Create the LANDFALLS DataFrame (one row per place visited):""";

LANDFALLS = pandas.DataFrame([
    # (voyage, date,        place,                   modern_country)
    (1, "1492-10-12", "San Salvador (Guanahani)", "Bahamas"),
    (1, "1492-10-28", "Cuba",                     "Cuba"),
    (1, "1492-12-05", "Hispaniola",               "Haiti"),
    (1, "1493-01-13", "Samana Peninsula",         "Dominican Republic"),
    (2, "1493-11-03", "Dominica",                 "Dominica"),
    (2, "1493-11-04", "Guadeloupe",               "France"),
    (2, "1493-11-14", "Saint Croix",              "United States"),
    (2, "1493-11-19", "Puerto Rico",              "United States"),
    (2, "1493-11-22", "Hispaniola",               "Haiti"),
    (2, "1494-05-05", "Jamaica",                  "Jamaica"),
    (3, "1498-07-31", "Trinidad",                 "Trinidad and Tobago"),
    (3, "1498-08-05", "Paria Peninsula",          "Venezuela"),
    (3, "1498-08-14", "Margarita",                "Venezuela"),
    (3, "1498-08-19", "Hispaniola",               "Dominican Republic"),
    (4, "1502-06-15", "Martinique",               "France"),
    (4, "1502-06-29", "Santo Domingo",            "Dominican Republic"),
    (4, "1502-07-30", "Guanaja",                  "Honduras"),
    (4, "1502-08-14", "Puerto Castilla",          "Honduras"),
    (4, "1502-10-16", "Almirante Bay",            "Panama"),
    (4, "1503-05-10", "Cayman Islands",           "United Kingdom"),
    (4, "1503-06-25", "St. Ann's Bay, Jamaica",   "Jamaica"),
], columns=["voyage", "date", "place", "modern_country"])

LANDFALLS["date"] = pandas.to_datetime(LANDFALLS["date"])
LANDFALLS.head()

> **Note:** This list shows only the *main* landfalls, not every island.
> `modern_country` shows who governs each place today. For example, Guadeloupe and Martinique are part of France,
> and Saint Croix and Puerto Rico are United States territories.

`value_counts()` counts how many times each value appears in a column.
`groupby("col")` splits the table into groups with the same value and then summarizes each group:

```python
LANDFALLS.groupby("voyage")["place"].count()
```

In [ ]:
"""EXERCISE 6.1: How many landfalls are listed for each voyage?""";

#TODO

In [ ]:
"""EXERCISE 6.2: Which modern countries appear most often? Use value_counts()""";

#TODO

In [ ]:
"""EXERCISE 6.3: Which places were visited on more than one voyage?
Hint: value_counts() on `place`, then keep only the counts > 1.""";

#TODO

In [ ]:
"""EXERCISE 6.4 (challenge): For each voyage, find the date of its first and last landfall.
Hint: groupby("voyage")["date"].agg(["min", "max"])""";

#TODO

## 7. Mapping the Landfalls

To put the landfalls on a map, each row needs a **latitude** and a **longitude**.
We can add a new column by assigning a list with one value per row, in the same order as the rows.

The coordinates below are approximate. They mark the bay or island where each landing took place.

In [ ]:
"""Run this cell to add approximate coordinates to LANDFALLS (same row order as above)""";

LANDFALLS["lat"] = [24.05, 21.10, 19.80, 19.20,                 # voyage 1
                    15.42, 16.25, 17.78, 18.28, 19.75, 18.47,   # voyage 2
                    10.10, 10.65, 11.00, 18.47,                 # voyage 3
                    14.70, 18.47, 16.45, 16.01, 9.30, 19.31, 18.44]   # voyage 4
LANDFALLS["lon"] = [-74.50, -75.73, -73.38, -69.40,
                    -61.35, -61.58, -64.76, -67.19, -72.20, -77.41,
                    -61.05, -62.60, -63.90, -69.89,
                    -61.18, -69.89, -85.90, -86.03, -82.25, -81.25, -77.20]
LANDFALLS.head()

We draw the map with [`plotly`](https://plotly.com/python/), which builds interactive charts.
`plotly.express.line_geo()` takes a DataFrame and the **names** of the columns to use,
just like the pandas functions we have seen:

* `lat=` and `lon=` set the position of each point.
* `color=` gives each voyage its own color. We also give each voyage its own marker shape (`symbol=`),
  so readers can tell the voyages apart without relying on color.
* `line_group=` connects the landfalls of the same voyage, in row order (which is date order).

**Move your mouse over a point** to see the place and date. You can also zoom and drag the map.

In [ ]:
"""Run this cell to draw the landfalls of each voyage on a map""";
import plotly.express as px

map_data = LANDFALLS.copy()
map_data["voyage"] = "Voyage " + map_data["voyage"].astype(str)   # text labels -> one color per voyage
map_data["date"] = map_data["date"].dt.strftime("%d %b %Y")

fig = px.line_geo(
    map_data, lat="lat", lon="lon",
    color="voyage", symbol="voyage", line_group="voyage",
    hover_name="place", hover_data={"date": True, "modern_country": True,
                                    "voyage": False, "lat": False, "lon": False},
    markers=True,
    color_discrete_sequence=["#2a78d6", "#eb6834", "#1baf7a", "#4a3aa7"],
    title="Main landfalls of Columbus's four voyages, 1492-1503",
)
fig.update_traces(line_width=2, marker_size=9)
fig.update_geos(
    projection_type="mercator", resolution=50,     # resolution=50 draws the small islands too
    lataxis_range=[7, 27], lonaxis_range=[-90, -57],
    showland=True, landcolor="#efeee9",
    showocean=True, oceancolor="#fcfcfb",
    showcountries=True, countrycolor="#c3c2b7", coastlinecolor="#a9a8a0",
)
fig.update_layout(height=520, margin=dict(l=0, r=0, t=50, b=0), legend_title_text="")
fig.show()

> **Note:** the lines join one landfall to the next. They are *not* the exact routes the ships sailed.

**QUESTIONS:**
* Which voyage reached the mainland of Central America? Which one reached South America?
* Hispaniola appears on several voyages. Can you see why it became the center of Spanish activity in the Caribbean?

In [ ]:
"""EXERCISE 7.1: Map only the first voyage.
Hint: filter map_data first, then pass the result to px.line_geo (or px.scatter_geo).""";

#TODO

## 8. Bonus: Combining Tables

`VOYAGES` and `SHIPS` share the column `voyage`.
`pandas.merge()` joins them on this column, much like a SQL `JOIN`:

In [ ]:
"""Run to attach each ship to the details of its voyage""";

pandas.merge(SHIPS, VOYAGES[["voyage", "depart_port", "depart_date"]], on="voyage")

**QUESTION:** Voyages 2 and 3 are missing from the result. Why?
(Hint: look up the `how=` argument of `pandas.merge`.)

<a id="sec-Excel"></a>
## 9. Try It in Excel

The three tables are also saved as CSV files in the `columbus/` folder, next to this notebook:

| File | One row per | Columns |
|---|---|---|
| `columbus/voyages.csv` | voyage (4) | `voyage`, `depart_date`, `return_date`, `depart_port`, `num_ships`, `first_landfall` |
| `columbus/ships.csv` | ship (7) | `voyage`, `ship`, `ship_type`, `captain`, `owner_or_master` (an unknown captain is a blank cell) |
| `columbus/landfalls.csv` | landfall (21) | `voyage`, `date`, `place`, `modern_country`, `lat`, `lon` |

Open them in Excel (or Google Sheets, or LibreOffice Calc) and answer the same questions you answered with pandas:

| # | Question | File | In Excel |
|---|---|---|---|
| 1 | Which ships were caravels? | `ships.csv` | Filter |
| 2 | Which ships have no known captain? | `ships.csv` | Filter: (Blanks) |
| 3 | How many days did each voyage last? Sort longest first. | `voyages.csv` | Formula, then Sort |
| 4 | How many landfalls on each voyage? | `landfalls.csv` | `COUNTIF` or PivotTable |
| 5 | Which modern country appears most often? | `landfalls.csv` | PivotTable |
| 6 | Which place appears on more than one voyage? | `landfalls.csv` | `COUNTIF` |
| 7 | Make a bar chart of the voyage durations. | `voyages.csv` | Insert › Bar chart |

> **Dates before 1900:** Excel cannot store dates before 1900, so the voyage dates open as text and cannot be subtracted.
> For question 3, add 800 years to both dates first (the leap years repeat every 400 years, so the number of days stays the same).
> With `depart_date` in column B and `return_date` in column C:
>
> ```
> =DATE(LEFT(C2,4)+800,MID(C2,6,2),RIGHT(C2,2)) - DATE(LEFT(B2,4)+800,MID(B2,6,2),RIGHT(B2,2))
> ```
>
> pandas has no such limit: `pandas.to_datetime()` handled 1492 without any workaround.

**QUESTION:** Which tool did you find easier for these questions? Which would you choose for a table with a million rows?

### Back to pandas: Reading a CSV File

`pandas.read_csv()` loads a CSV file into a `DataFrame`. You will use it in every session from now on.

In [ ]:
"""Run this cell to load ships.csv. What happened to the blank captain cells?""";

SHIPS_CSV = pandas.read_csv("columbus/ships.csv")
SHIPS_CSV

In [ ]:
"""EXERCISE 9.1: Load columbus/landfalls.csv and check your Excel answer to question 5.
Hint: read_csv, then value_counts() on `modern_country`.""";

#TODO

**QUESTION:** Print the `dtypes` of the DataFrame you just loaded. Is `date` a date again, or text?
What would you need to do before computing durations?

---

**Takeaway:** these few commands make up most everyday data exploration:
`DataFrame(...)`, `read_csv`, `head`, `shape`, `dtypes`, `df[...]`, `.loc`, boolean filters,
`isna`, `to_datetime`, `sort_values`, `value_counts`, `groupby` and `merge`.
In Session 1, we will use the same tools on real system data.

---
## Solutions

Try the exercises before you look here!

In [ ]:
"""Suggested solutions""";

# 2.1
print(VOYAGES.shape)
# 2.2
print(VOYAGES.dtypes)
# 3.1
print(VOYAGES[["voyage", "num_ships"]])
# 3.2
print(VOYAGES.loc[2])
# 4.1
print(SHIPS[SHIPS["ship_type"] == "caravel"])
# 4.2
print(SHIPS[SHIPS["captain"] == "Christopher Columbus"])
# 4.3
print(SHIPS.isna().sum())
# 4.4
print(SHIPS[SHIPS["captain"].isna()])
# 5.1  (dates were already converted in the cell above)
VOYAGES["duration_days"] = (VOYAGES["return_date"] - VOYAGES["depart_date"]).dt.days
# 5.2
print(VOYAGES.sort_values("duration_days", ascending=False))
# 5.3
d = VOYAGES.sort_values("num_ships")
fig, ax = pyplot.subplots(figsize=(8, 3.5))
bars = ax.barh("Voyage " + d["voyage"].astype(str), d["num_ships"], height=0.55, color="#2a78d6")
ax.bar_label(bars, padding=6)
ax.set_title("Number of ships per voyage", loc="left")
pyplot.show()
# 6.1
print(LANDFALLS.groupby("voyage")["place"].count())
# 6.2
print(LANDFALLS["modern_country"].value_counts())
# 6.3
counts = LANDFALLS["place"].value_counts()
print(counts[counts > 1])
# 6.4
print(LANDFALLS.groupby("voyage")["date"].agg(["min", "max"]))
# 7.1
first = map_data[map_data["voyage"] == "Voyage 1"]
fig = px.line_geo(first, lat="lat", lon="lon", hover_name="place", markers=True,
                  title="Landfalls of the first voyage, 1492-1493")
fig.update_geos(lataxis_range=[15, 27], lonaxis_range=[-80, -66], showland=True)
fig.show()
# 9.1
LANDFALLS_CSV = pandas.read_csv("columbus/landfalls.csv")
print(LANDFALLS_CSV["modern_country"].value_counts())

**Answers to the Excel questions (Section 9):**

1. Pinta, Nina, Capitana, Gallega, Vizcaina and Santiago de Palos (6 caravels; only the Santa Maria is a carrack).
2. Gallega, Vizcaina and Santiago de Palos.
3. Voyage 2: 991 days, voyage 4: 910, voyage 3: 815, voyage 1: 213.
4. Voyage 1: 4, voyage 2: 6, voyage 3: 4, voyage 4: 7.
5. Dominican Republic, 3 times.
6. Hispaniola, 3 times.
7. The chart should match the one in Section 5.